# Segment a long recording into clips for transcription

Takes **one long audio file** (your ~1-hour code-switched recording) and cuts it
at natural pauses into short **3–14 second clips** + a `manifest.csv`. Then you
download the clips and transcribe them in the **Setswana Transcription Tool**
(`transcription/transcribe_app/index.html`) — hear one clip, type what you hear, Enter.

No GPU needed. Nothing is uploaded anywhere except your own Google Drive.


## 1. Mount Drive + point at your audio file

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS: path to your long recording on Drive (wav / mp3 / m4a / ogg all fine)
AUDIO_FILE = '/content/drive/MyDrive/voiceai_tts/raw/my_recording.m4a'

import os
assert os.path.exists(AUDIO_FILE), f'Not found: {AUDIO_FILE}  <-- fix the path'
print('Found:', AUDIO_FILE, '(', round(os.path.getsize(AUDIO_FILE)/1e6,1), 'MB )')


## 2. Install audio tools

In [ ]:
!apt-get -qq install -y ffmpeg >/dev/null
!pip -q install pydub
print('ready')


## 3. Cut on silence into 3–14 second clips

Tune only if needed:
- clips come out **too long / run-on** -> raise `SILENCE_THRESH_ADJ` toward 0 (e.g. -12)
- clips get **chopped mid-word** -> lower it (e.g. -20), or raise `MIN_SILENCE_MS`.


In [ ]:
import math, csv
from pydub import AudioSegment, silence

MIN_SILENCE_MS   = 400    # a pause this long counts as a cut point
SILENCE_THRESH_ADJ = -16  # dB below the track's average loudness = 'silence'
MIN_CLIP_MS = 3000        # merge anything shorter into its neighbour
MAX_CLIP_MS = 14000       # hard-split anything longer
PAD_MS = 120              # keep a little air around each clip
SR = 22050                # matches the Piper training pipeline

print('Loading + normalising audio...')
audio = AudioSegment.from_file(AUDIO_FILE).set_channels(1).set_frame_rate(SR)
total_min = len(audio)/60000
print(f'Length: {total_min:.1f} min, avg loudness {audio.dBFS:.1f} dBFS')

ranges = silence.detect_nonsilent(audio, min_silence_len=MIN_SILENCE_MS,
                                  silence_thresh=audio.dBFS + SILENCE_THRESH_ADJ, seek_step=10)
print('Raw speech ranges found:', len(ranges))

# pack adjacent ranges up to MAX
packed = []
for s, e in ranges:
    if packed and e - packed[-1][0] <= MAX_CLIP_MS:
        packed[-1][1] = e
    else:
        packed.append([s, e])
# merge too-short into previous
merged = []
for s, e in packed:
    if merged and (e - s) < MIN_CLIP_MS and (e - merged[-1][0]) <= MAX_CLIP_MS * 1.4:
        merged[-1][1] = e
    else:
        merged.append([s, e])
# hard-split anything still over MAX
final = []
for s, e in merged:
    if e - s <= MAX_CLIP_MS:
        final.append((s, e))
    else:
        n = math.ceil((e - s) / MAX_CLIP_MS); step = (e - s) / n
        for k in range(n):
            final.append((int(s + k*step), int(s + (k+1)*step)))

durs = [(e-s)/1000 for s, e in final]
print(f'\n=> {len(final)} clips, {sum(durs)/60:.1f} min of speech, '
      f'avg {sum(durs)/len(durs):.1f}s (min {min(durs):.1f}s / max {max(durs):.1f}s)')


## 4. Write the clips + manifest, zip, and download

In [ ]:
import shutil
OUT = '/content/tsn_clips'
if os.path.exists(OUT): shutil.rmtree(OUT)
os.makedirs(OUT, exist_ok=True)

rows = []
for j, (s, e) in enumerate(final, 1):
    a = max(0, s - PAD_MS); b = min(len(audio), e + PAD_MS)
    cid = f'tsn_{j:04d}'
    audio[a:b].export(f'{OUT}/{cid}.wav', format='wav')
    rows.append((cid, a, b, round((b-a)/1000, 2)))

with open(f'{OUT}/manifest.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['id', 'start_ms', 'end_ms', 'dur_s']); w.writerows(rows)

# save a copy to Drive (safe) and a zip you can download
DRIVE_OUT = '/content/drive/MyDrive/voiceai_tts/clips_to_transcribe'
os.makedirs(os.path.dirname(DRIVE_OUT) + '/', exist_ok=True)
if os.path.exists(DRIVE_OUT): shutil.rmtree(DRIVE_OUT)
shutil.copytree(OUT, DRIVE_OUT)
zip_path = shutil.make_archive('/content/tsn_clips', 'zip', OUT)
print('Zip size:', round(os.path.getsize(zip_path)/1e6, 1), 'MB  (', len(rows), 'clips )')
print('Also saved to Drive:', DRIVE_OUT)

from google.colab import files
files.download(zip_path)


## Next: transcribe

1. Unzip `tsn_clips.zip` on your PC.
2. Open `transcription/transcribe_app/index.html`, click **Choose clip files**, select all the `.wav` clips.
3. Hear a clip -> type exactly what you hear (Setswana + English, as spoken) -> **Enter**. Progress auto-saves.
4. When done (or as you go), click **Export CSV** -> `metadata.csv`.
5. That `metadata.csv` + the clips drop straight into the Piper training notebook (same LJSpeech format).